# Downloading historical data

There is no way to do a historical filter with SP IQ so I have had to create a big universe (7,500 stocks) and download their market cap and number of estimates at each date. Then we can filter for our universe back in time

In [3]:
import pandas as pd
import duckdb
import os

con = duckdb.connect("research.duckdb")

In [ ]:
cwd = os.getcwd()
filepath = rf"{cwd}\datsets\fundamentals\all_stocks_and_dates.csv"

def create_universe_db(filepath):
    universe_df = pd.read_csv(filepath, encoding="cp1252")

    con.execute("DROP TABLE IF EXISTS universe")

    con.execute("""
    CREATE TABLE universe AS
    SELECT AS_OF, company_name, IQID FROM universe_df
    """).df()

# create_universe_db(filepath)

Cross join dates and tickers to create row for every ticker at every date. Populate a database with the excel commands to get the data from CIQ

In [8]:
def cross_join_create_formulas():
    '''
    This takes the table with all of the tickers and all of the dates
    and performs a cross join    
    '''
    con.execute("DROP TABLE IF EXISTS historical_eps_formula")
    con.execute(
        """
    CREATE OR REPLACE TABLE historical_eps_formula AS
    WITH
    unique_dates AS (
        SELECT DISTINCT strptime(AS_OF, '%d/%m/%Y')::DATE AS AS_OF 
        FROM universe
        -- CRITICAL: Remove nulls and unparseable dates
        WHERE AS_OF IS NOT NULL 
        AND AS_OF != 'NaT'
        AND AS_OF != ''
    ),
    unique_assets AS (
        SELECT DISTINCT company_name, IQID 
        FROM universe
        -- Optional: Ensure you don't have blank company names
        WHERE company_name IS NOT NULL
    )

    SELECT
        d.AS_OF,
        a.company_name,
        a.IQID,
        CONCAT('=@SPG(', a.IQID, ', \"SP_EXCHANGE\", \"', d.AS_OF, '\")') AS exchange,
        CONCAT('=@SPG(', a.IQID, ', \"IQ_SECTOR\", \"', d.AS_OF, '\")') AS sector,
        CONCAT('=SPG(', a.IQID, ', \"SP_MARKETCAP\", \"', d.AS_OF, '\")') AS marketcap,
        CONCAT('=SPG(', a.IQID, ', \"SP_EPS_NUM_EST\", \"FY+1\", \"', d.AS_OF, '\")') AS num_est,
        CONCAT('=SPG(', a.IQID, ', \"SP_EPS_EST\", \"FY+1\", \"', d.AS_OF, '\")') AS eps_est,
        CONCAT('=SPG(', a.IQID, ', \"SP_EST_EPS_SURPRISE_PERCENT\", \"FY0\", \"', d.AS_OF, '\")') AS surprise
    FROM unique_dates d
    CROSS JOIN unique_assets a
    ORDER BY a.company_name, d.AS_OF

    """
    )

cross_join_create_formulas()

In [ ]:
def create_excel_sheets():
    '''
    splits the whole cross joined database into 5000 row sections
    this seemed to be the limit for pulling historical data on excel
    '''
    count = con.execute("SELECT COUNT() FROM historical_ciq_formula").df().iloc[0,0]
    batch_size = 5000
    n_batches = (count // batch_size) + 1
    for i in range(n_batches):
        con.execute(f"COPY (SELECT * FROM historical_ciq_formula LIMIT 5000 OFFSET {i*batch_size}) TO 'datsets/fundamentals/csv_batches/test_formula_{i}.csv' (HEADER)")

automatically refresh data

In [23]:
import pandas as pd
import os
SOURCE_DIR = r'C:\Users\adamf\PycharmProjects\FedContracts_AlgoTrading\datsets\fundamentals\csv_batches'
OUTPUT_DIR = r'C:\Users\adamf\PycharmProjects\FedContracts_AlgoTrading\datsets\fundamentals\csv_batches_reloaded'

completed = [c for c in os.listdir(OUTPUT_DIR) if c.endswith('.csv')]

print(len(completed))

save_path = f'{OUTPUT_DIR}/{completed[0]}'
df = pd.read_csv(save_path)
df

14


,AS_OF,company_name,IQID,exchange,sector,marketcap,num_est,eps_est,surprise
0,01/09/2014,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,NASDAQGS,Consumer Discretionary,330.05,4.0,0.23,9.09
1,01/10/2014,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,NASDAQGS,Consumer Discretionary,488.57,1.0,0.26,-4.35
2,01/11/2014,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,NASDAQGS,Consumer Discretionary,513.51,4.0,0.47,-4.35
3,01/12/2014,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,NASDAQGS,Consumer Discretionary,514.67,4.0,0.46,-4.35
4,01/01/2015,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,NASDAQGS,Consumer Discretionary,528.79,4.0,0.46,-4.35
...,...,...,...,...,...,...,...,...,...
4995,01/02/2020,A. O. Smith Corporation (NYSE:AOS),4991081,NYSE,Industrials,NaN,NaN,NaN,8.89
4996,01/02/2020,A. O. Smith Corporation (NYSE:AOS),4020605,NYSE,Industrials,"6,963.25",10.0,2.42,-1.77
4997,01/03/2020,A. O. Smith Corporation (NYSE:AOS),4991081,NYSE,Industrials,NaN,NaN,NaN,8.89
4998,01/03/2020,A. O. Smith Corporation (NYSE:AOS),4020605,NYSE,Industrials,"6,405.95",12.0,2.33,-1.77


In [ ]:

def excel_to_table():
    OUTPUT_DIR = r'C:\Users\adamf\PycharmProjects\FedContracts_AlgoTrading\datsets\fundamentals\csv_batches_reloaded\*'

    con.execute(f"""
                CREATE OR REPLACE TABLE master_historical AS
                SELECT * FROM read_csv_auto('{OUTPUT_DIR}')
    """)

In [7]:
def check_errors():
    df = con.execute("""
        SELECT DISTINCT IQID, num_est
        FROM master_historical
        WHERE TRY_CAST(replace(num_est, ',', '') AS FLOAT) IS NULL 
        AND num_est IS NOT NULL
        AND num_est NOT IN ('NA', 'N/A', '', 'NaN')
    """).df()

    return df

check_errors()

,IQID,num_est


In [8]:
con.execute("""
            CREATE OR REPLACE TABLE master_historical AS 
            SELECT * REPLACE (
                TRY_CAST(REPLACE(marketcap, ',', '') AS FLOAT) AS marketcap,
                TRY_CAST(REPLACE(num_est, ',', '') AS INT) AS num_est,
                TRY_CAST(REPLACE(eps_est, ',', '') AS FLOAT) AS eps_est,
                TRY_CAST(REPLACE(surprise, ',', '') AS FLOAT) AS surprise
            )
            FROM master_historical
""")

In [9]:
con.execute("DESCRIBE SELECT * FROM master_historical LIMIT 10").df()

,column_name,column_type,null,key,default,extra
0,AS_OF,DATE,YES,None,None,None
1,company_name,VARCHAR,YES,None,None,None
2,IQID,BIGINT,YES,None,None,None
3,exchange,VARCHAR,YES,None,None,None
4,sector,VARCHAR,YES,None,None,None
5,marketcap,FLOAT,YES,None,None,None
6,num_est,INTEGER,YES,None,None,None
7,eps_est,FLOAT,YES,None,None,None
8,surprise,FLOAT,YES,None,None,None


In [10]:
con.execute("ALTER TABLE master_historical RENAME TO master_historical_eps")
con.execute("SHOW TABLES").df()

,name
0,daily_extended_universe
1,developments
2,eps_features
3,eps_raw
4,extended_universe
5,historical_ciq_formula
6,historical_price_ciq_formula
7,master_historical_eps
8,master_historical_prices
9,universe
